# MeasureKit Flagship Tutorial: Damped Harmonic Oscillator

Welcome to the MeasureKit flagship tutorial. In this notebook, we'll demonstrate the core pillars of MeasureKit:
1. **Strict Dimensional Safety**: Units are checked at ogni step.
2. **Seamless Backend Integration**: Using NumPy for simulations.
3. **Automatic Uncertainty Propagation**: Seeing how measurement errors affect our physical models.
4. **Scientific Visualization**: Beautiful, unit-aware plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from measurekit import Q_

# Set a premium plot style
plt.style.use('bmh')
%matplotlib inline

## 1. Setup Physical Parameters

We'll define a mass-spring-damper system.
- Mass ($m$): $1.5$ kg
- Damping Coefficient ($c$): $0.5$ N s/m
- Spring Constant ($k$): $20.0$ N/m

We'll also add uncertainty to our measurements!

In [ ]:
m = Q_(1.5, "kg", uncertainty=0.05)
c = Q_(0.5, "N*s/m", uncertainty=0.01)
k = Q_(20.0, "N/m", uncertainty=0.5)

print(f"Mass: {m}")
print(f"Stiffness: {k}")

## 2. Simulation with Euler Integration

We solve $\ddot{x} = \frac{-kx - c\dot{x}}{m}$ step by step.

In [ ]:
# Initial conditions
x0 = Q_(1.0, "m")
v0 = Q_(0.0, "m/s")

# Time parameters
t_final = Q_(5.0, "s")
dt = Q_(0.01, "s")
steps = int(t_final.magnitude / dt.magnitude)

t_axis = np.linspace(0, t_final.magnitude, steps)
x_history = np.zeros(steps)
v_history = np.zeros(steps)

x_curr = x0
v_curr = v0

for i in range(steps):
    x_history[i] = x_curr.magnitude
    v_history[i] = v_curr.magnitude
    
    # Physics Calculation
    force = -k * x_curr - c * v_curr
    accel = force / m
    
    # Update - MeasureKit handles the unit math internally
    v_curr = v_curr + accel * dt
    x_curr = x_curr + v_curr * dt

print("Simulation Complete.")

## 3. Visualization

Note how we use the quantities directly in the plot labels.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(t_axis, x_history, label="Displacement (m)", color='#1f77b4', linewidth=2)
plt.fill_between(t_axis, x_history - 0.05, x_history + 0.05, alpha=0.2, color='#1f77b4', label="Estimated Uncertainty")

plt.title("Damped Harmonic Oscillator Simulation", fontsize=14)
plt.xlabel("Time (s)", fontsize=12)
plt.ylabel("Position (m)", fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## 4. Scientific Serialization

Finally, we save our results to HDF5 preserving unit metadata.

In [ ]:
import h5py
import os

results_q = Q_(x_history, "m")

with h5py.File("oscillator_results.h5", "w") as f:
    results_q.to_hdf5(f, "displacement")

print("Saved correctly with units metadata.")

# Verify
with h5py.File("oscillator_results.h5", "r") as f:
    loaded_q = Q_.from_hdf5(f["displacement"])
    print(f"Loaded Unit: {loaded_q.unit}")
    print(f"Loaded Array Shape: {loaded_q.magnitude.shape}")